# Camera-ready revision experiments (DeSE 2026)
Self-contained: needs only `Training-data.csv` in the same folder. Run top to bottom. Results are saved as CSV files as it runs.

Addresses: repeated splits + mean/std (R1 #8, R2), paired test MLP vs GB (R2), equal tuning budget (R1 #5), train-only thresholds/scaling (R1 #4), architecture comparison for the depth claim (R1 #7), MLP convergence info (R1 #6).

In [1]:
import os, tempfile
# Optional: keep joblib temp files on D: (same as your thesis notebook)
d_drive_temp = "D:\\Thesis_Temp_Storage"
if os.path.exists("D:\\"):
    os.makedirs(d_drive_temp, exist_ok=True)
    os.environ["JOBLIB_TEMP_FOLDER"] = d_drive_temp
    tempfile.tempdir = d_drive_temp
    print("Temp storage ->", d_drive_temp)

Temp storage -> D:\Thesis_Temp_Storage


## CELL R1 — Setup & leak-free feature pipeline

In [2]:
import time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

SEED = 42
raw = pd.read_csv("Training-data.csv").dropna()

def row_features(d):
    """Row-wise features only (no statistics from other rows -> no leakage)."""
    d = d.copy()
    t = d["Tenure"] + 1
    d["Avg_Spend_Per_Tenure"] = np.log1p(d["Total Spend"] / t)
    d["Usage_Per_Tenure"]     = np.log1p(d["Usage Frequency"] / t)
    d["Support_Call_Rate"]    = d["Support Calls"] / t
    d["Payment_Delay_Rate"]   = np.log1p(d["Payment Delay"] / t)   # log applied ONCE
    d["Recency_Score"]        = 1 / (d["Last Interaction"] + 1)
    d["Engagement_Score"]     = d["Usage_Per_Tenure"] - d["Support_Call_Rate"] - d["Payment_Delay_Rate"]
    d["Age_Group"] = pd.cut(d["Age"], bins=[18,25,35,45,55,65,100],
                            labels=["18-25","26-35","36-45","46-55","56-65","65+"])
    d = pd.get_dummies(d, columns=["Gender","Subscription Type","Contract Length","Age_Group"],
                       drop_first=True)
    d = d.drop(columns=["CustomerID","Tenure","Usage Frequency","Total Spend",
                        "Last Interaction","Age"])
    # 'Support Calls' and 'Payment Delay' kept for now: HighRisk needs them, then drops them
    return d.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

class HighRisk(BaseEstimator, TransformerMixin):
    """High_Risk flag with medians learned from the TRAINING fold only."""
    def fit(self, X, y=None):
        self.m_calls_ = X["Support Calls"].median()
        self.m_delay_ = X["Payment Delay"].median()
        return self
    def transform(self, X):
        X = X.copy()
        X["High_Risk"] = ((X["Support Calls"] > self.m_calls_) &
                          (X["Payment Delay"] > self.m_delay_)).astype(float)
        return X.drop(columns=["Support Calls", "Payment Delay"])

data = row_features(raw)
X_all = data.drop(columns="Churn")
y_all = data["Churn"].astype(int)

# Split off a small TUNING set that is never used in the final evaluation
X_eval, X_tune, y_eval, y_tune = train_test_split(
    X_all, y_all, test_size=40_000, stratify=y_all, random_state=SEED)
print("Evaluation set:", X_eval.shape, "| Tuning set:", X_tune.shape)

def make_pipe(model):
    # Same preprocessing for every model; fitted inside each training fold only
    return Pipeline([("risk", HighRisk()), ("scale", StandardScaler()), ("clf", model)])

Evaluation set: (400832, 18) | Tuning set: (40000, 18)


## CELL R2 — Equal tuning budget for every model

In [3]:
N_ITER, TUNE_CV = 10, 3          # identical budget: 10 random configs x 3-fold CV per model

search_spaces = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=SEED),
        {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}),
    "Naive Bayes": (GaussianNB(),
        {"clf__var_smoothing": np.logspace(-11, -5, 10)}),
    "KNN": (KNeighborsClassifier(n_jobs=-1),
        {"clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
         "clf__weights": ["uniform", "distance"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
        {"clf__max_depth": [4, 6, 8, 10, 12, 16, None],
         "clf__min_samples_leaf": [1, 10, 50, 100]}),
    "Random Forest": (RandomForestClassifier(n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [100, 200, 300],
         "clf__max_depth": [8, 12, 16, None],
         "clf__min_samples_leaf": [1, 10, 50],
         "clf__max_features": ["sqrt", 0.5]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=SEED),
        {"clf__n_estimators": [100, 200, 300],
         "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
         "clf__max_depth": [3, 4, 5, 6],
         "clf__subsample": [0.8, 1.0]}),
    "MLP": (MLPClassifier(solver="adam", max_iter=300, early_stopping=True,
                          validation_fraction=0.1, n_iter_no_change=10, random_state=SEED),
        {"clf__hidden_layer_sizes": [(50,), (100,), (200,), (100, 50), (128, 64), (100, 50, 25)],
         "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
         "clf__learning_rate_init": [1e-3, 3e-3]}),
}

best_params, tuning_rows = {}, []
for name, (model, space) in search_spaces.items():
    t0 = time.time()
    n_iter = min(N_ITER, int(np.prod([len(v) for v in space.values()])))
    rs = RandomizedSearchCV(make_pipe(model), space, n_iter=n_iter, cv=TUNE_CV,
                            scoring="accuracy", random_state=SEED, n_jobs=1)
    rs.fit(X_tune, y_tune)
    best_params[name] = rs.best_params_
    tuning_rows.append({"Model": name, "Configs tried": n_iter,
                        "Best CV acc": round(rs.best_score_, 4),
                        "Best params": rs.best_params_,
                        "Minutes": round((time.time() - t0) / 60, 1)})
    if name == "MLP":            # keep every architecture's score for the depth question
        mlp_search_results = pd.DataFrame(rs.cv_results_)[
            ["param_clf__hidden_layer_sizes", "param_clf__alpha",
             "param_clf__learning_rate_init", "mean_test_score", "std_test_score"]
        ].sort_values("mean_test_score", ascending=False)
    print(f"{name:20s} best={rs.best_score_:.4f}  {rs.best_params_}")

tuning_table = pd.DataFrame(tuning_rows)
tuning_table.to_csv("rev_tuning_table.csv", index=False)
mlp_search_results.to_csv("rev_mlp_architectures.csv", index=False)
display(tuning_table)
display(mlp_search_results)

Logistic Regression  best=0.8919  {'clf__C': 100}
Naive Bayes          best=0.7663  {'clf__var_smoothing': np.float64(1e-05)}
KNN                  best=0.8913  {'clf__weights': 'distance', 'clf__n_neighbors': 3}
Decision Tree        best=0.9369  {'clf__min_samples_leaf': 10, 'clf__max_depth': 12}
Random Forest        best=0.9527  {'clf__n_estimators': 100, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': None}
Gradient Boosting    best=0.9606  {'clf__subsample': 1.0, 'clf__n_estimators': 200, 'clf__max_depth': 6, 'clf__learning_rate': 0.1}
MLP                  best=0.9548  {'clf__learning_rate_init': 0.003, 'clf__hidden_layer_sizes': (100, 50), 'clf__alpha': 0.0001}


,Model,Configs tried,Best CV acc,Best params,Minutes
0,Logistic Regression,6,0.8919,{'clf__C': 100},0.1
1,Naive Bayes,10,0.7663,{'clf__var_smoothing': 1e-05},0.1
2,KNN,10,0.8913,"{'clf__weights': 'distance', 'clf__n_neighbors...",0.7
3,Decision Tree,10,0.9369,"{'clf__min_samples_leaf': 10, 'clf__max_depth'...",0.2
4,Random Forest,10,0.9527,"{'clf__n_estimators': 100, 'clf__min_samples_l...",1.1
5,Gradient Boosting,10,0.9606,"{'clf__subsample': 1.0, 'clf__n_estimators': 2...",14.9
6,MLP,10,0.9548,"{'clf__learning_rate_init': 0.003, 'clf__hidde...",7.1


,param_clf__hidden_layer_sizes,param_clf__alpha,param_clf__learning_rate_init,mean_test_score,std_test_score
7,"(100, 50)",0.00010,0.003,0.954850,0.002047
0,"(100,)",0.00100,0.003,0.953150,0.001151
3,"(100, 50)",0.01000,0.003,0.952600,0.003520
8,"(200,)",0.00001,0.001,0.951900,0.001627
5,"(50,)",0.01000,0.003,0.951075,0.001640
2,"(100,)",0.00100,0.001,0.950100,0.003625
1,"(200,)",0.01000,0.001,0.950025,0.001034
9,"(50,)",0.00100,0.003,0.949825,0.000161
6,"(50,)",0.00010,0.001,0.949425,0.001091
4,"(50,)",0.00100,0.001,0.949325,0.001099


## CELL R3 — 5x2 repeated cross-validation

In [4]:
# 5 repetitions x 2 folds = 10 train/test evaluations per model, same splits for every model
def tuned_model(name):
    model, _ = search_spaces[name]
    return make_pipe(clone(model)).set_params(**best_params[name])

rows, fold_acc = [], {name: [] for name in search_spaces}
for rep in range(5):
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=rep)
    for fold, (tr, te) in enumerate(skf.split(X_eval, y_eval)):
        Xtr, Xte = X_eval.iloc[tr], X_eval.iloc[te]
        ytr, yte = y_eval.iloc[tr], y_eval.iloc[te]
        for name in search_spaces:
            t0 = time.time()
            pipe = tuned_model(name).fit(Xtr, ytr)
            prob = pipe.predict_proba(Xte)[:, 1]
            pred = (prob >= 0.5).astype(int)
            acc = accuracy_score(yte, pred)
            fold_acc[name].append(acc)
            row = {"Model": name, "rep": rep, "fold": fold, "Accuracy": acc,
                   "Precision": precision_score(yte, pred), "Recall": recall_score(yte, pred),
                   "F1": f1_score(yte, pred), "ROC_AUC": roc_auc_score(yte, prob),
                   "fit_sec": time.time() - t0}
            if name == "MLP":
                row["mlp_epochs"] = pipe.named_steps["clf"].n_iter_
            rows.append(row)
            print(f"rep {rep} fold {fold} | {name:20s} acc={acc:.4f} ({row['fit_sec']:.0f}s)")
        pd.DataFrame(rows).to_csv("rev_5x2_raw_results.csv", index=False)  # saved as it goes

results = pd.DataFrame(rows)
summary = (results.groupby("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
           .agg(["mean", "std"]).sort_values(("Accuracy", "mean"), ascending=False))
summary.to_csv("rev_summary_mean_std.csv")
display(summary.round(4))

rep 0 fold 0 | Logistic Regression  acc=0.8946 (1s)
rep 0 fold 0 | Naive Bayes          acc=0.7636 (1s)
rep 0 fold 0 | KNN                  acc=0.9228 (101s)
rep 0 fold 0 | Decision Tree        acc=0.9503 (3s)
rep 0 fold 0 | Random Forest        acc=0.9620 (11s)
rep 0 fold 0 | Gradient Boosting    acc=0.9666 (354s)
rep 0 fold 0 | MLP                  acc=0.9631 (134s)
rep 0 fold 1 | Logistic Regression  acc=0.8949 (2s)
rep 0 fold 1 | Naive Bayes          acc=0.8078 (1s)
rep 0 fold 1 | KNN                  acc=0.9221 (117s)
rep 0 fold 1 | Decision Tree        acc=0.9509 (3s)
rep 0 fold 1 | Random Forest        acc=0.9615 (14s)
rep 0 fold 1 | Gradient Boosting    acc=0.9659 (347s)
rep 0 fold 1 | MLP                  acc=0.9635 (52s)
rep 1 fold 0 | Logistic Regression  acc=0.8939 (1s)
rep 1 fold 0 | Naive Bayes          acc=0.8055 (1s)
rep 1 fold 0 | KNN                  acc=0.9222 (97s)
rep 1 fold 0 | Decision Tree        acc=0.9497 (4s)
rep 1 fold 0 | Random Forest        acc=0.9615 (11

Accuracy         Precision          Recall          \
                        mean     std      mean     std    mean     std   
Model                                                                    
Gradient Boosting     0.9662  0.0005    0.9862  0.0004  0.9538  0.0006   
MLP                   0.9632  0.0005    0.9858  0.0030  0.9487  0.0028   
Random Forest         0.9619  0.0004    0.9839  0.0003  0.9483  0.0006   
Decision Tree         0.9508  0.0006    0.9805  0.0009  0.9318  0.0012   
KNN                   0.9228  0.0005    0.9661  0.0003  0.8954  0.0010   
Logistic Regression   0.8948  0.0006    0.9418  0.0008  0.8682  0.0012   
Naive Bayes           0.7856  0.0195    0.9757  0.0045  0.6379  0.0382   

                         F1         ROC_AUC          
                       mean     std    mean     std  
Model                                                
Gradient Boosting    0.9697  0.0004  0.9921  0.0001  
MLP                  0.9669  0.0004  0.9907  0.0001  
Random Forest        0.9658  0.0003  0.9901  0.0002  
Decision Tree        0.9555  0.0006  0.9828  0.0003  
KNN                  0.9294  0.0005  0.9606  0.0004  
Logistic Regression  0.9035  0.0006  0.9514  0.0003  
Naive Bayes          0.7708  0.0266  0.9131  0.0009

## CELL R4 — Paired statistical tests

In [5]:
def paired_ttest_5x2cv(a, b):
    """Dietterich (1998) 5x2cv paired t-test. a, b = 10 fold scores in rep/fold order."""
    d = np.array(a).reshape(5, 2) - np.array(b).reshape(5, 2)
    s2 = ((d - d.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
    t = d[0, 0] / np.sqrt(s2.mean())
    p = 2 * stats.t.sf(abs(t), df=5)
    return t, p

best = summary.index[0]
test_rows = []
for other in summary.index[1:]:
    t, p = paired_ttest_5x2cv(fold_acc[best], fold_acc[other])
    w = stats.wilcoxon(fold_acc[best], fold_acc[other]).pvalue
    test_rows.append({"Comparison": f"{best} vs {other}",
                      "Mean diff (acc)": np.mean(fold_acc[best]) - np.mean(fold_acc[other]),
                      "5x2cv t": t, "5x2cv p": p, "Wilcoxon p": w})
tests = pd.DataFrame(test_rows)
tests.to_csv("rev_paired_tests.csv", index=False)
display(tests.round(4))

# The specific comparison Reviewer 2 asked for
t, p = paired_ttest_5x2cv(fold_acc["MLP"], fold_acc["Gradient Boosting"])
print(f"MLP vs Gradient Boosting: mean diff = "
      f"{np.mean(fold_acc['MLP']) - np.mean(fold_acc['Gradient Boosting']):.4f}, "
      f"5x2cv t = {t:.3f}, p = {p:.4f}")
print("MLP epochs to convergence (per fold):",
      results.loc[results.Model == "MLP", "mlp_epochs"].tolist())

,Comparison,Mean diff (acc),5x2cv t,5x2cv p,Wilcoxon p
0,Gradient Boosting vs MLP,0.0031,6.2799,0.0015,0.002
1,Gradient Boosting vs Random Forest,0.0043,15.3698,0.0000,0.002
2,Gradient Boosting vs Decision Tree,0.0154,25.9821,0.0000,0.002
3,Gradient Boosting vs KNN,0.0434,55.7904,0.0000,0.002
4,Gradient Boosting vs Logistic Regression,0.0714,113.6322,0.0000,0.002
5,Gradient Boosting vs Naive Bayes,0.1806,7.7482,0.0006,0.002


MLP vs Gradient Boosting: mean diff = -0.0031, 5x2cv t = -6.280, p = 0.0015
MLP epochs to convergence (per fold): [44.0, 25.0, 30.0, 39.0, 36.0, 44.0, 32.0, 32.0, 38.0, 32.0]


## CELL R5 — (Optional) Keras ANN on the same splits

In [7]:
# OPTIONAL — Keras ANN on the SAME 10 splits (needs TensorFlow). Skip if short on time.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

keras_rows = []
for rep in range(5):
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=rep)
    for fold, (tr, te) in enumerate(skf.split(X_eval, y_eval)):
        tf.keras.utils.set_random_seed(SEED + rep)           # fixed seed per run
        prep = Pipeline([("risk", HighRisk()), ("scale", StandardScaler())])
        Xtr = prep.fit_transform(X_eval.iloc[tr]); ytr = y_eval.iloc[tr].values
        Xte = prep.transform(X_eval.iloc[te]);     yte = y_eval.iloc[te].values
        model = Sequential([Input(shape=(Xtr.shape[1],)),
                            Dense(128, activation="relu", kernel_initializer="glorot_uniform"),
                            Dropout(0.2),
                            Dense(64, activation="relu"),
                            Dense(1, activation="sigmoid")])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                      loss="binary_crossentropy", metrics=["accuracy"])
        es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        h = model.fit(Xtr, ytr, epochs=100, batch_size=1024, validation_split=0.1,
                      callbacks=[es], verbose=0)
        prob = model.predict(Xte, verbose=0).ravel(); pred = (prob >= 0.5).astype(int)
        keras_rows.append({"Model": "Keras ANN", "rep": rep, "fold": fold,
                           "Accuracy": accuracy_score(yte, pred),
                           "Precision": precision_score(yte, pred), "Recall": recall_score(yte, pred),
                           "F1": f1_score(yte, pred), "ROC_AUC": roc_auc_score(yte, prob),
                           "epochs_run": len(h.history["loss"])})
        print(f"rep {rep} fold {fold} | Keras acc={keras_rows[-1]['Accuracy']:.4f} "
              f"epochs={keras_rows[-1]['epochs_run']}")

keras_res = pd.DataFrame(keras_rows)
keras_res.to_csv("rev_keras_5x2.csv", index=False)
display(keras_res[["Accuracy","Precision","Recall","F1","ROC_AUC"]].agg(["mean","std"]).round(4))

rep 0 fold 0 | Keras acc=0.9641 epochs=48
rep 0 fold 1 | Keras acc=0.9637 epochs=42
rep 1 fold 0 | Keras acc=0.9636 epochs=38
rep 1 fold 1 | Keras acc=0.9640 epochs=38
rep 2 fold 0 | Keras acc=0.9640 epochs=49
rep 2 fold 1 | Keras acc=0.9635 epochs=48
rep 3 fold 0 | Keras acc=0.9633 epochs=38
rep 3 fold 1 | Keras acc=0.9618 epochs=31
rep 4 fold 0 | Keras acc=0.9632 epochs=43
rep 4 fold 1 | Keras acc=0.9638 epochs=36


,Accuracy,Precision,Recall,F1,ROC_AUC
mean,0.9635,0.9856,0.9495,0.9672,0.9909
std,0.0007,0.0017,0.0022,0.0006,0.0003
